# Gold Employee Aggregation

This notebook builds gold-layer employee aggregates from the curated silver datasets.

* Load standardized silver employee data from CSV, JSON, and XML pipelines
* Align schemas across sources before combining them
* Create gold-level employee and department summary aggregations
* Persist the results as Delta data and Unity Catalog tables
* Validate the final gold outputs for reporting use cases


## Gold Employee Aggregation

This section combines all silver employee sources, creates employee-level aggregate metrics by source, department, gender, and country, and stores the gold result.


In [0]:
# Load silver employee datasets from CSV, JSON, and XML pipelines, standardize their schemas,
# combine them into one dataset, build gold-level employee aggregates, and validate the output.
from pyspark.sql import functions as F

silver_employee_path = "abfss://landing@bronzeadlsstorage.dfs.core.windows.net/silver/employee/"
silver_employee_json_path = "abfss://landing@bronzeadlsstorage.dfs.core.windows.net/silver/employee_json/"
silver_employee_xml_path = "abfss://landing@bronzeadlsstorage.dfs.core.windows.net/silver/employee_xml/"
gold_employee_path = "abfss://landing@bronzeadlsstorage.dfs.core.windows.net/gold/employee_aggregation/"

silver_employee_df = (
    spark.read.format("delta").load(silver_employee_path)
    .select(
        F.col("Employee_id").cast("long").alias("Employee_id"),
        F.col("Full_Name"),
        F.col("Gender"),
        F.col("Country"),
        F.col("Currency"),
        F.col("Department_id").cast("long").alias("Department_id"),
        F.col("Manager_id").cast("long").alias("Manager_id"),
        F.col("Salary").cast("double").alias("Salary"),
        F.col("Age").cast("long").alias("Age"),
        F.col("Date_of_Joining").alias("Date_of_Joining"),
        F.lit("csv").alias("Source_Type")
    )
)

silver_employee_json_df = (
    spark.read.format("delta").load(silver_employee_json_path)
    .select(
        F.col("Employee_id").cast("long").alias("Employee_id"),
        F.col("Full_Name"),
        F.col("Gender"),
        F.col("Country"),
        F.col("Currency"),
        F.col("Department").cast("long").alias("Department_id"),
        F.col("Manager_id").cast("long").alias("Manager_id"),
        F.col("Salary").cast("double").alias("Salary"),
        F.col("Age").cast("long").alias("Age"),
        F.to_date(F.col("Date_Of_Joining"), "dd/MM/yyyy").alias("Date_of_Joining"),
        F.lit("json").alias("Source_Type")
    )
)

silver_employee_xml_df = (
    spark.read.format("delta").load(silver_employee_xml_path)
    .select(
        F.col("Employee_id").cast("long").alias("Employee_id"),
        F.col("Full_Name"),
        F.col("Gender"),
        F.col("Country"),
        F.col("Currency"),
        F.col("Department").cast("long").alias("Department_id"),
        F.col("Manager_id").cast("long").alias("Manager_id"),
        F.col("Salary").cast("double").alias("Salary"),
        F.col("Age").cast("long").alias("Age"),
        F.to_date(F.col("Date_Of_Joining"), "dd/MM/yyyy").alias("Date_of_Joining"),
        F.lit("xml").alias("Source_Type")
    )
)

all_employees_df = (
    silver_employee_df
    .unionByName(silver_employee_json_df)
    .unionByName(silver_employee_xml_df)
)

gold_employee_agg_df = (
    all_employees_df
    .groupBy("Source_Type", "Department_id", "Gender", "Country")
    .agg(
        F.countDistinct("Employee_id").alias("employee_count"),
        F.avg("Salary").alias("avg_salary"),
        F.sum("Salary").alias("total_salary"),
        F.avg("Age").alias("avg_age"),
        F.min("Date_of_Joining").alias("first_joining_date"),
        F.max("Date_of_Joining").alias("latest_joining_date"),
        F.countDistinct("Manager_id").alias("manager_count")
    )
    .orderBy("Source_Type", "Department_id", "Gender", "Country")
)

gold_employee_agg_df.write \
    .mode("overwrite") \
    .format("delta") \
    .save(gold_employee_path)

final_gold_employee_agg_df = spark.read.format("delta").load(gold_employee_path)

display(final_gold_employee_agg_df)

In [0]:
# Save the employee gold aggregation as a managed Delta table and preview the registered table output.
gold_employee_summary_table = "adw2.default.gold_employee_summary"

final_gold_employee_agg_df.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable(gold_employee_summary_table)

employee_summary_table_df = spark.table(gold_employee_summary_table)

display(employee_summary_table_df)

## Gold Employee Summary Table

This section publishes the employee gold aggregation as a managed table for downstream analytics and reporting.


## Gold Department Summary Table

This section creates and publishes department-level gold aggregates as a managed table for reporting and analysis.


In [0]:
# Build department-level gold metrics from the combined employee dataset, save them as a Delta table,
# and validate the final department summary output.
gold_department_summary_table = "adw2.default.gold_department_summary"

gold_department_summary_df = (
    all_employees_df
    .groupBy("Source_Type", "Department_id")
    .agg(
        F.countDistinct("Employee_id").alias("employee_count"),
        F.avg("Salary").alias("avg_salary"),
        F.sum("Salary").alias("total_salary"),
        F.avg("Age").alias("avg_age"),
        F.countDistinct("Country").alias("country_count"),
        F.countDistinct("Manager_id").alias("manager_count"),
        F.min("Date_of_Joining").alias("first_joining_date"),
        F.max("Date_of_Joining").alias("latest_joining_date")
    )
    .orderBy("Source_Type", "Department_id")
)

gold_department_summary_df.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable(gold_department_summary_table)

department_summary_table_df = spark.table(gold_department_summary_table)

display(department_summary_table_df)